# Línea futuro · la entrega final (M30_30)

**Entrega final: `salidas/submissions/sub_M30_30.csv`** (`M30_30`): mezcla en probabilidades de dos LightGBM LambdaRank con
30 semillas, `0.7 · C30 + 0.3 · N30`, cada uno con el segundo paso de sesión. **CV 0.929 · público (controlador 020) 0.953**,
la mejor nota pública del equipo.

**Resumen en cinco líneas**
1. El reto: en cada snapshot (la lista de ~36 tareas abiertas del controlador aéreo en un instante) hay que poner arriba la
   tarea en la que está trabajando. Métrica: acierto top-3 (azar ≈ 0.09, ejemplo oficial 0.352).
2. El organizador permitió usar **cualquier información de los datasets**: snapshots pasados y futuros de la sesión y
   etiquetas de otras sesiones. Esta línea lo aprovecha (la que solo usa el pasado está en `realista.ipynb`).
3. **El futuro lo es casi todo:** con solo el snapshot se llega a ~0.66; con el pasado, ~0.67; con el futuro, ~0.90. El
   controlador decide y *después* actúa: la siguiente instrucción que da al avión delata la tarea.
4. Encima, tres mejoras sólidas (`clear`, el segundo paso de sesión, `instr` + árboles más grandes) llevan a **N** (0.924), y
   el **oráculo** (qué hicieron otros controladores en el mismo escenario) a **C** (0.928). Mezclados en probabilidades: 0.929.
5. La entrega se eligió por su **privado esperado** (controlador 023), no por el público, con cinco métodos distintos.

**Cómo usar este notebook.** Solo necesita la carpeta `entrega/` y los CSV de Kaggle (`train.csv`, `validation.csv`,
`test.csv`) copiados en `entrega/datos/`. **No entrena nada:** lee las salidas ya calculadas de `entrega/salidas/` y calcula las
tablas con `codigo/comun.py`. La primera vez construye las features en `entrega/cache/` (~1–3 min). Las cifras que no se
pueden recalcular desde esas salidas (modelos intermedios cuyos OOF no se guardaron) se citan de `notebooks/00_destilado.ipynb`
y `futuro/LGB_futuro.ipynb` del repo, y se marcan con **(citado)**.

| Sección | Qué cuenta |
|---|---|
| 0 | Las técnicas, explicadas desde cero |
| 1 | Los datos: qué se elige y la estructura de las etiquetas |
| 2 | De dónde sale la nota: la cadena de CV y cuánto azar hay |
| 3 | Por qué funciona lo que funciona |
| 4 | Dónde falla y dónde está el techo |
| 5 | Qué no funcionó |
| 6 | Público frente a privado y elección de la entrega |
| 7 | Comprobación de la entrega y cómo reproducirla |

In [ ]:
# Localiza la carpeta entrega/ (subiendo desde el directorio actual) y añade entrega/codigo al path
import sys
from pathlib import Path
import numpy as np, pandas as pd

ENTREGA = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "entrega" / "codigo" / "comun.py").exists()
               or (p / "codigo" / "comun.py").exists())
ENTREGA = ENTREGA if (ENTREGA / "codigo" / "comun.py").exists() else ENTREGA / "entrega"
sys.path.insert(0, str(ENTREGA / "codigo"))
import comun
from comun import labels, hits, medidas, leer, CV023, TIPOS

pd.set_option("display.width", 200, "display.max_columns", 30)
LAB = labels()                                         # sample_id -> target, atco, session, scenario, fold


def oof(nombre):
    o = leer(nombre, "oof")
    assert o is not None, f"falta salidas/oof/{nombre}.csv"
    return o


def tabla(nombres, ref=None):
    # Medidas de cada OOF (y diferencia frente a ref): CV, CV 023, peor controlador, mediana por sesión, escenarios
    r = oof(ref) if ref else None
    filas = []
    for n in nombres:
        m = medidas(oof(n), r if (ref and n != ref) else None)
        f = dict(modelo=n, cv=m["cv"], cv_023=m["cv_023"], peor_controlador=m["peor_controlador"],
                 mediana_sesion=m["mediana_sesion"], esc1=m["escenario_1"], esc2=m["escenario_2"], esc3=m["escenario_3"])
        if "delta" in m:
            f.update(delta_puntos=100 * m["delta"], sesiones_suben_bajan=f"{m['sesiones_suben']}/{m['sesiones_bajan']}",
                     controladores_suben_bajan=f"{m['controladores_suben']}/{m['controladores_bajan']}")
        filas.append(f)
    return pd.DataFrame(filas).set_index("modelo").round(4)


def por_controlador(nombres, ref):
    # Δ (puntos) del acierto por controlador frente a ref; * = los 6 parecidos al 023
    h0 = hits(oof(ref))
    t = {n: (hits(oof(n)).reindex(h0.index) - h0).groupby(LAB.atco.loc[h0.index]).mean() * 100 for n in nombres}
    t = pd.DataFrame(t).T.round(1)
    t.columns = [f"{a:03d}{'*' if a in CV023 else ''}" for a in t.columns]
    return t


print(f"entrega/ en {ENTREGA} · {len(LAB)} snapshots etiquetados · {LAB.atco.nunique()} controladores · "
      f"salidas OOF: {sorted(p.stem for p in (ENTREGA / 'salidas' / 'oof').glob('*.csv'))}")

## 0 · Las técnicas, explicadas desde cero

**El problema como *ranking*.** En cada snapshot hay una lista de ~36 tareas y hay que **ordenarlas** de más a menos probable.
No es una clasificación "sí/no" fila a fila: importa la posición de la tarea correcta *dentro de su lista*. Por eso la métrica
es el **top-3 hit rate**: la fracción de snapshots en los que la tarea correcta queda entre las 3 primeras.

**Validación cruzada por controlador, OOF y CV anidada.**
- **CV por controlador** (`datos/folds.csv`, 5 folds fijos): todos los snapshots de un controlador caen en el mismo fold. El test
  es de controladores nuevos (020 y 023), así que la CV mide lo mismo: acertar con un controlador que el modelo nunca ha visto.
  Un reparto aleatorio daría una nota inflada (snapshots a 3 s del mismo controlador comparten respuesta).
- **OOF (*out-of-fold*):** la predicción de cada snapshot hecha por el modelo del fold que no lo vio. Con los OOF se comparan y
  combinan modelos sin hacer trampa.
- **CV anidada:** cuando además hay que *elegir* algo (un peso, un parámetro), se elige con 4 folds y se mide en el 5.º.
  Elegirlo mirando todo el OOF da una cifra optimista.
- **CV 023:** el acierto en los 6 controladores de la CV más parecidos al 023 del privado (002, 011, 010, 006, 017, 015), elegidos
  por variables que se ven sin etiquetas.

**LightGBM LambdaRank.** *Gradient boosting*: cientos de árboles pequeños en serie, cada uno corrige lo que los anteriores aún
hacen mal. LightGBM maneja de forma nativa las celdas vacías (aquí significan "no aplica") y las categorías (`task_index`).
**LambdaRank** optimiza directamente el orden dentro de cada snapshot, con más peso en las primeras posiciones. **63 hojas** y
`min_child_samples = 100`: árboles más expresivos sin hojas diminutas. **`extra_trees`**: los cortes se eligen al azar entre
unos pocos candidatos; los árboles salen más distintos y el conjunto generaliza mejor a controladores nuevos. **Semillas**: se
entrena cada fold con k semillas y se promedian (±0.2 puntos de CV con 3 semillas; casi nada a partir de ~20).

**Softmax y segundo paso de sesión.** El softmax convierte los scores de un snapshot en probabilidades que suman 1. El
**segundo paso** es un logit condicional (softmax dentro del snapshot) de solo 9 pesos que usa el score del primer modelo en
*toda la sesión* (por ejemplo, la probabilidad máxima que esa tarea tuvo en otras rachas). Al ser tan pequeño no sobreajusta.

**El oráculo.** Todas las sesiones de un escenario tienen el mismo tráfico. El oráculo mira, en las otras sesiones etiquetadas del
mismo escenario y en el mismo momento, si esa tarea fue la respuesta. Usa etiquetas de otras sesiones, así que se calcula **por
fold**: para predecir un fold solo cuentan sesiones de los otros folds y de otros controladores.

**Mezcla en probabilidades.** M30_30 = 0.7 · softmax(C) + 0.3 · softmax(N). Si un modelo da ~0 a una tarea, el otro aún puede
salvarla (en log-probabilidades, un ~0 la vetaría). El peso 0.3 se eligió por CV anidada.

**Familias de variables** (`codigo/features_futuro.py` y `codigo/oraculo.py`):

| Familia | Prefijo | Qué mide | Usa |
|---|---|---|---|
| base | (originales) | tipo de tarea, tiempos al borde del sector, estado del avión, rangos `l_*` | solo el snapshot |
| logic | `logic_*` | reglas escritas a mano: plazo por tipo, si la tarea es factible ahora, orden de urgencia | solo el snapshot |
| past | `p_*`, `blk_*`, `gap_*` | cuánto lleva la tarea en la lista, qué cambió desde el snapshot anterior, posición en la racha | snapshots pasados |
| future | `f_*` | segundos hasta la siguiente instrucción al avión y su tipo; cuándo desaparece la tarea | snapshots futuros |
| clear | `sx_*` | segundos hasta que cambia el nivel autorizado, el rumbo, la zona, el conflicto o la ruta del avión | snapshots futuros |
| instr | `si_*` | segundos hasta la siguiente instrucción **de cada tipo** y hasta la que **encaja** con la tarea | snapshots futuros |
| oráculo | `or_*` | qué hicieron otros controladores con esa tarea en el mismo escenario | otras sesiones |

*Vocabulario:* **racha** = snapshots seguidos de una sesión sin huecos de más de 4.5 s; **tramo** = racha × tarea correcta.

## 1 · Los datos

Cada fila es una tarea del snapshot; exactamente una tiene `target_index = 1`. Train + validation: 18 controladores,
37 sesiones, 3 escenarios de tráfico. Test: controladores 020 (público) y 023 (privado, el que decide).

In [ ]:
from features_futuro import construir, BLOQUES
lab, te, fs = construir()                     # la primera vez calcula y guarda entrega/cache/features_futuro.parquet
resp = lab[lab.target_index == 1].set_index("sample_id")
snap = lab.drop_duplicates("sample_id").set_index("sample_id")[["session", "scenario", "atco", "fold", "block", "t"]]
print(" · ".join(f"{b} = {len(fs[b])}" for b in BLOQUES) + f" · total {sum(len(fs[b]) for b in BLOQUES)} columnas")
pd.DataFrame({
    "snapshots": [lab.sample_id.nunique(), te.sample_id.nunique()],
    "filas": [len(lab), len(te)],
    "sesiones": [lab.session.nunique(), te.session.nunique()],
    "controladores": [lab.atco.nunique(), te.atco.nunique()],
    "tareas por snapshot (media)": [round(lab.groupby("sample_id").size().mean(), 1),
                                    round(te.groupby("sample_id").size().mean(), 1)],
}, index=["etiquetado", "test"])

### 1a · Qué tipos de tarea se eligen: el *lift*

Lift = % de las respuestas / % de las filas. Transfer y assume son pocas filas y muchas respuestas; las coordinaciones
(entry/exit) son muchas filas y pocas respuestas: casi siempre hay alguna en la lista y casi nunca es la que se trabaja.

In [ ]:
filas = lab.task_index.value_counts(normalize=True)
resps = resp.task_index.value_counts(normalize=True)
lift = pd.DataFrame({"% filas": filas * 100, "% respuestas": resps * 100}).fillna(0)
lift["lift"] = lift["% respuestas"] / lift["% filas"]
lift.index = [TIPOS[i] for i in lift.index]
lift.sort_values("% respuestas", ascending=False).round(2)

### 1b · La estructura de las etiquetas

Dos regularidades que no son de un snapshot sino de la sesión, y que explican buena parte del modelo final:
1. **Dentro de una racha el controlador sigue con la misma tarea.** Tras un hueco, casi nunca vuelve a ella.
2. **Cada tarea se trabaja en un solo episodio de la sesión.** Es lo que aprovecha el segundo paso de sesión (sección 3b).

In [ ]:
s = snap.join(resp[["task_name"]]).sort_values(["session", "t"])
s["prev_task"] = s.groupby("session").task_name.shift()
s["prev_block"] = s.groupby("session").block.shift()
dentro = s[s.block == s.prev_block]
tras_hueco = s[(s.block != s.prev_block) & s.prev_block.notna()]
ep = s.groupby(["session", "task_name"]).block.nunique()
pd.Series({
    "repite la respuesta del snapshot anterior, dentro de la racha": f"{(dentro.task_name == dentro.prev_task).mean():.1%}",
    "repite la respuesta tras un hueco entre rachas": f"{(tras_hueco.task_name == tras_hueco.prev_task).mean():.1%}",
    "tareas que son respuesta en una sola racha de la sesión": f"{(ep == 1).mean():.1%}",
    "snapshots por racha (media)": f"{s.groupby(['session', 'block']).size().mean():.1f}",
}).to_frame("valor")

**Lectura** (cifras del destilado, **citado**): dentro de una racha se repite la respuesta el 94.8 % de las veces; tras un hueco,
el 0.2 %; el 98.4 % de las tareas son respuesta en una sola racha.

### 1c · La señal principal, sin modelo: P(elegida) según los segundos hasta la siguiente instrucción

`f_next_input` = segundos hasta la siguiente instrucción que recibe el avión (cualquiera); `si_match` = segundos hasta la
siguiente instrucción que **encaja** con el tipo de tarea (nivel → exit_conditions, rumbo → return_to_route o zone,
transfer → transfer…), aunque no sea la primera.

In [ ]:
import matplotlib.pyplot as plt
cortes = [-1, 3, 10, 20, 30, 60, 120, 300, 1e9]
et = ["≤3", "3–10", "10–20", "20–30", "30–60", "60–120", "120–300", ">300"]
curva = pd.DataFrame({nombre: lab.target_index.groupby(pd.cut(lab[var], cortes, labels=et), observed=False).mean() * 100
                      for var, nombre in (("f_next_input", "cualquier instrucción al avión"),
                                          ("si_match", "instrucción que encaja con la tarea"))})
fig, ax = plt.subplots(figsize=(7, 3.6))
for c in curva:
    ax.plot(range(len(et)), curva[c].values, marker="o", ms=5, lw=2, label=c)
base = lab.target_index.mean() * 100
ax.axhline(base, color="grey", lw=1, ls="--")
ax.text(len(et) - 1, base + 0.6, "media de una tarea cualquiera", ha="right", color="grey", fontsize=8)
ax.set_xticks(range(len(et)), et); ax.set_xlabel("segundos hasta la siguiente instrucción")
ax.set_ylabel("% de filas que son la elegida")
ax.set_title("Cuanto antes llega la instrucción, más probable es que esa sea la tarea", loc="left")
ax.legend(frameon=False); plt.tight_layout(); plt.show()
curva.round(1).T

## 2 · De dónde sale la nota

### 2a · La cadena de CV

Todos los pasos medidos con los mismos folds. `limpia` usa solo el propio snapshot; `real_adri_noche`, también el pasado (es la
entrega de `realista.ipynb`); desde `futuro_base`, el futuro de la sesión.

| Paso | Modelo | Qué añade |
|---|---|---|
| 1 | `limpia` | LightGBM LambdaRank, solo el snapshot (`lgb_adri_v2` en el repo) |
| 2 | `real_adri_noche` | + snapshots pasados de la sesión |
| 3 | `futuro_base` | + bloque `future` (siguiente instrucción, desaparición de la tarea) |
| 4 | `futuro_clear` | + bloque `clear` (cambios futuros de nivel, rumbo, zona, ruta) |
| 5 | `futuro_clear_ses` | + segundo paso de sesión |
| 6 | `A` | 10 semillas + suavizado en la racha |
| 7 | `N10_ses` | + bloque `instr`, 63 hojas, `extra_trees` (= **N**) |
| 8 | `C10_ses` | N + oráculo de coordinaciones (= **NOR** / **C**) |
| 9 | `M30_30` | 0.7 · C30 + 0.3 · N30 en probabilidades (**entrega**) |

In [ ]:
CADENA = ["limpia", "real_adri_noche", "futuro_base", "futuro_clear", "futuro_clear_ses", "A", "N10_ses", "C10_ses", "M30_30"]
t = tabla(CADENA)
t["Δ_paso_puntos"] = (t.cv.diff() * 100).round(2)
t

**Lectura** (cifras del destilado y de `EQUIPO.md`, **citado**):
- **El pasado aporta ~1.5 puntos** (0.656 → 0.672). **El futuro aporta ~24** (0.672 → 0.90). Casi toda la nota sale de ver qué
  hace el controlador *después*.
- Sobre el futuro, tres mejoras sólidas: `clear` (+0.6), el **segundo paso de sesión** (+0.7) e `instr` + árboles más grandes
  (+0.6). Ninguna de las dos piezas de la última gana por separado; juntas, sí.
- El **oráculo** suma +0.4 en CV sobre N, pero es la única fuente cuyo efecto varía mucho de un controlador a otro (sección 6).

### 2b · Cuánto azar hay: semillas y reproducibilidad

El mismo modelo con 3, 10 y 30 semillas; y `C10_otra_maquina_ses`, la **misma configuración** que `C10_ses` (N + oráculo,
10 semillas) entrenada por separado en otra máquina.

In [ ]:
tabla(["N3_ses", "N10_ses", "N30_ses", "C3_ses", "C10_ses", "C10_otra_maquina_ses", "C30_ses"])

In [ ]:
# ¿Cuánto cambia el top-3 de test (023) al cambiar de semillas o de máquina, frente a cambiar de modelo?
t3 = lambda n: comun.topk(leer(n, "preds")).apply(lambda r: frozenset(r))
ids023 = te.loc[te.atco == 23, "sample_id"].unique()
pares = [("N3_ses", "N10_ses"), ("N10_ses", "N30_ses"), ("C10_ses", "C10_otra_maquina_ses"), ("C10_ses", "C30_ses"),
         ("N30_ses", "C30_ses"), ("A", "N10_ses"), ("N30_ses", "M30_30")]
pd.DataFrame([dict(a=a, b=b, top3_distinto_023=f"{(t3(a).reindex(ids023) != t3(b).reindex(ids023)).mean():.1%}")
              for a, b in pares]).set_index(["a", "b"])

**Lectura** (**citado** del destilado, sección 2b):
- **Folds:** con los folds fijos, la nota de N va de 0.907 a 0.957 según el fold: ±2 puntos según qué controladores tocan. Es la
  misma incertidumbre que tendrá el privado, que es un solo controlador.
- **Semillas:** la CV casi no se mueve con el número de semillas (N: 0.9233 / 0.9242 / 0.9234 con 3 / 10 / 30). Por
  controlador, de 10 a 30 semillas el acierto cambia ±0.5 puntos.
- **Modelo frente a azar:** cambiar de modelo mueve muchos más snapshots del 023 (N → N + oráculo: ~34 %) que cambiar de
  semillas (~6–13 %): las diferencias entre los modelos adoptados no son azar.
- **Reproducibilidad:** la misma configuración en otra máquina da la misma CV (0.9278 frente a 0.9280) y el mismo top-3 en ~86 %
  de los snapshots, lo mismo que cambiar de grupo de semillas.

## 3 · Por qué funciona lo que funciona

### 3a · La regla que aprende el modelo: "el avión que recibe una instrucción en los próximos 10 s"

Contribución (SHAP del primer pase, en %) de cada origen de información (`futuro/pruebas/f9_explica.py`). M0 = sin `clear`,
A = con `clear`, O = con el oráculo completo, B = A + oráculo de coordinaciones. El reparto es el mismo en CV, en el test del
020 y en el del 023: el modelo no se comporta distinto con los controladores de test.

In [ ]:
pd.read_csv(ENTREGA / "salidas" / "notas" / "f9_influencia.csv", index_col=[0, 1]).round(1)

| Siguiente instrucción al avión | ≤ 3 s | 3–10 s | 10–30 s | 30–60 s | > 60 s |
|---|---|---|---|---|---|
| efecto medio sobre el score (SHAP, modelo A) **(citado)** | +2.0 | +1.9 | +0.9 | ≈ 0 | −0.1 |

### 3b · El segundo paso de sesión (`codigo/sesion.py`)

Un logit condicional de 9 pesos sobre el score del primer pase en las otras rachas de la sesión. El peso más grande es
**negativo** (M ≈ −2.6, casi igual en los 5 folds): **si la tarea ya fue la favorita en otra racha, aquí probablemente no lo es**
("explaining away"), porque cada tarea se trabaja una sola vez (sección 1b). Tiene que ser pequeño: un LightGBM en su lugar
sobreajusta (0.906 frente a 0.914, **citado**).

In [ ]:
# Efecto del segundo paso con el mismo primer pase
tabla(["futuro_clear", "futuro_clear_ses"], ref="futuro_clear")

### 3c · `instr` y árboles de 63 hojas con `extra_trees`

`f_next_input` solo mira la **primera** instrucción siguiente; en el 16 % de las filas la instrucción que encaja con la tarea no es
la primera (**citado**). El bloque `instr` mide, para cada tipo de instrucción, cuándo llega la siguiente y cuándo llega la que
encaja con la tarea. Con árboles pequeños no aporta; con 63 hojas y `extra_trees`, sí (+0.6 juntos, ninguno por separado).

### 3d · El oráculo de coordinaciones y por qué se mezcla en probabilidades

El oráculo (`codigo/oraculo.py`, variante *coord*: solo en `entry_coordination` y `exit_coordination`) dice si otros
controladores trabajaron esa coordinación en el mismo momento del mismo escenario. Entre sesiones se transmite *si* una tarea
se trabaja, no *cuándo*. Sobre N suma ~+0.4 de CV, pero en algunos controladores resta: un controlador que coordina distinto
que los demás (el 006) perdía mucho cuando la mezcla se hacía en log, porque el oráculo vetaba sus coordinaciones raras.
Mezclado en probabilidades con N (0.7 · C + 0.3 · N), N rescata esas tareas.

In [ ]:
tabla(["N30_ses", "C30_ses", "M30_30"], ref="N30_ses")

In [ ]:
# Δ por controlador (puntos) frente a N30_ses (* = los 6 parecidos al 023)
por_controlador(["C30_ses", "M30_30"], ref="N30_ses")

## 4 · Dónde falla y dónde está el techo

In [ ]:
# Acierto top-3 según el tipo de la tarea correcta
def por_tipo(nombre):
    h = hits(oof(nombre))
    return h.groupby(resp.task_index.reindex(h.index).map(lambda i: TIPOS[i])).mean()

P = pd.DataFrame({n: por_tipo(n) for n in ("limpia", "A", "N30_ses", "M30_30")})
P["% de respuestas"] = resp.task_index.map(lambda i: TIPOS[i]).value_counts(normalize=True) * 100
P.sort_values("% de respuestas", ascending=False).round(3)

In [ ]:
# Los fallos son de tramo entero: si el modelo falla un tramo, lo falla en todos sus snapshots
h = hits(oof("M30_30")).rename("hit").to_frame().join(snap).join(resp.task_name)
tr = h.groupby(["session", "block", "task_name"]).hit.agg(["mean", "size"])
fallos = (1 - h.hit).sum()
pd.Series({
    "acierto (M30_30)": f"{h.hit.mean():.4f}",
    "tramos acertados enteros": f"{(tr['mean'] == 1).mean():.1%}",
    "tramos fallados enteros": f"{(tr['mean'] == 0).mean():.1%}",
    "% de los fallos que están en tramos fallados enteros": f"{(tr.loc[tr['mean'] == 0, 'size'].sum() / fallos):.1%}",
}).to_frame("M30_30")

**Lo que dicen los diagnósticos** (**citado**, `futuro/pruebas/f8_diag.py`, `f8_parejas.py`, `errores.py`):
- **La mitad de los fallos son coordinaciones** (entry/exit). Coordinar con un sector vecino no es una instrucción al avión: no
  deja huella en las columnas. El 77 % de los tramos fallados no tiene ninguna huella observable ni en el pasado ni en el futuro.
- **El 41 % de los fallos acierta el avión pero no la tarea.** En las tareas de acción la respuesta desaparece a los ~3 s; en las
  coordinaciones la señal se invierte (desaparece la hermana y la coordinación sigue 1–3 min).
- **Los fallos son de tramo entero** (~80 % de los fallos): ningún post-proceso en la racha puede arreglarlos, porque no hay un
  snapshot bueno del que tirar.
- **Ante la misma situación, dos controladores eligen la misma tarea solo el 12.7 % de las veces.** Parte de la decisión es
  preferencia personal o información que no está en los datos.
- **La curva de aprendizaje no se aplana** (25/50/75/100 % de los controladores: 0.870 / 0.893 / 0.898 / 0.906): más
  controladores ayudarían.

## 5 · Qué no funcionó

Todo medido con la misma CV y las mismas semillas que la referencia; detalle en `futuro/LGB_futuro.ipynb` (LF) (**citado**).

| Familia | Pruebas | Resultado | Por qué |
|---|---|---|---|
| Otros modelos | XGBoost, CatBoost, red (transformer) con futuro | 0.896 / 0.874 / 0.887 frente a 0.901 (LF 5, 6c) | con la información del futuro todos aprenden lo mismo y fallan en los mismos snapshots |
| Stacks y mezclas sin oráculo | media ponderada, meta-ranker | ±0.1 (LF 6c, 13c, 15) | los modelos están muy correlacionados; los pesos van a uno solo |
| Parámetros | más lento, más regularizado, más o menos hojas, objetivos binary / xendcg | ruido (LF 2–3, 12a) | manda la información, no el ajuste |
| Bloques de features | `coord`, `block`, `rel`, `geo` | 0 a −0.4 (LF 6, 11a, 16, 18, 19a) | lo que dicen ya está en `f_task_end` o en los rangos |
| Post-procesos en la racha | HMM, Kalman, modelo por racha, transiciones aprendidas | = suavizado o peor (LF 15, 23b) | los tramos se aciertan o se fallan enteros |
| Variantes del segundo paso | Sinkhorn, LightGBM, más variables | saturado (LF 9, 11b) | el logit de 9 pesos ya recoge la estructura |
| Adaptación al controlador | prior por EM, covariate shift, Bayes jerárquico, estilo, pseudo-etiquetas | ruido o peor (LF 11c, 15, 18b, 21a, 23) | el estilo del controlador no es estable fuera de muestra |
| Especialistas | solo coordinaciones, solo tareas con instrucción | peso óptimo 0 (LF 16, 23a) | no saben cuándo la respuesta NO es de su tipo |

### 5b · Lo que no deja huella: ¿un experto sin futuro lo arregla?

En ~41 % de los snapshots **ningún candidato tiene una instrucción compatible en ≤ 30 s** (se ve sin etiquetas, también en
test). Ahí el modelo pierde su pista principal. Idea: que en esos snapshots decidan las prioridades (plazos, cercanía al borde,
antigüedad), que es lo que aprendieron los modelos **sin futuro** (`limpia`, `real_adri_noche`). Se mezclan solo en esos
snapshots, con el peso elegido por CV anidada (`codigo/sinpista.py`, ~1 min).

In [ ]:
import sinpista
T_sinpista = sinpista.main()

**Lectura** (**citado** del destilado, sección 5b): la CV anidada da muy poco peso al experto (0.1–0.3) y el resultado queda
dentro del ruido entre semillas. Las prioridades ya están dentro del modelo principal. Cuando no hay huella, **ni el futuro, ni el
pasado, ni las prioridades, ni lo que hicieron otros controladores** distinguen la tarea correcta: esa información no está en
los datos (comunicación no registrada con sectores vecinos, preferencia del controlador o ruido del etiquetado).

## 6 · Público frente a privado y elección de la entrega

### 6a · El público engaña

Kaggle solo enseña el público (controlador 020, 2 sesiones). Se elige **una** entrega y la clasificación final es **solo el
privado** (controlador 023). El organizador avisó de un fuerte sobreajuste al público en la competición.

| Envío | CV | Público (020) |
|---|---|---|
| A | 0.918 | 0.947 |
| B = A + oráculo | 0.925 | 0.934 |
| N | 0.924 | 0.951 |
| NOR = N + oráculo | 0.928 | 0.948 |
| **M30_30** | **0.929** | **0.953** |

Los primeros envíos con oráculo (sobre A) bajaron en el público aunque la CV los daba por mejores; sobre N ya no. La hipótesis
de que el 020 castigaba al oráculo porque coordina mucho dejó de cumplirse con los envíos siguientes: el público son 2 sesiones y
±1–2 puntos de ruido. El 023, en cambio, casi no tiene coordinaciones como respuesta.

### 6b · Elegir la entrega por su privado esperado (`codigo/eleccion.py`, ~2 min)

Todo como diferencia frente a **N** (la de mejor público conocido antes de M30_30). Cinco formas distintas de estimar el privado:
1. **Validar la regla de decisión:** cada uno de los 18 controladores de la CV hace por turnos de "privado"; se elige la
   candidata con mejor CV en los otros 17 y se mide en él.
2. **Riesgo a la baja por controlador:** media de los 3 peores, peor controlador, P(perder > 1 punto frente a N).
3. **Cota por desacuerdo en test:** dos entregas solo pueden sacar notas distintas en los snapshots en que su top-3 difiere.
4. **CV reponderada hacia el 023:** cada snapshot de la CV pesa según cuánto se parece a los del 023 en variables sin etiqueta
   (regresión logística CV frente a 023).
5. **Señal sin etiquetas validada:** solo vale si en la CV predice la diferencia de acierto entre modelos.

**Privado esperado** = media de los métodos 2 y 4. El público se informa, pero no entra en la decisión.

In [ ]:
import eleccion
riesgo, info = eleccion.main()

**Lectura y decisión** (**citado** del destilado, sección 6b–6c):
- **Regla validada:** elegir por CV da M30_30 en 16 de 18 controladores y gana +0.22 a "quedarse siempre con N".
- **Riesgo:** M30_30 no pierde más de 1 punto frente a N en ningún controlador y su peor controlador está por encima del de N.
- **Desacuerdo:** M30_30 cambia el top-3 de N en ~22 % de los snapshots del 023, menos que NOR o C30: es la opción con oráculo
  que menos se aleja de N.
- **Reponderada al 023:** las tres con oráculo ganan ~+0.6 a N; A pierde.
- **Señal sin etiquetas:** no predice el acierto en la CV (|r| ≤ 0.2): descartada.
- **Decisión: M30_30** (+0.54 frente a N; C30 +0.50 y NOR +0.49 quedan dentro del ruido entre semillas).

### 6c · Lo que sabemos del 023 (sin etiquetas)
- **Coordina poco:** el top-1 de los modelos es una coordinación en el 3–4 % de sus snapshots (~10 % en la CV).
- **Es atípico:** 023sc2 es la sesión de estilo más extrema de todas (muchos assume forzados); 023sc1 se parece al 001, el peor
  controlador de la CV.
- **Más difícil:** los modelos están menos seguros y hay menos snapshots con instrucción inminente: se esperaba un privado de
  ~0.91–0.92 (±5), unos 3 puntos por debajo del público, sea cual sea la entrega.
- En los 6 controladores de la CV más parecidos (CV 023), M30_30 da 0.934 frente a 0.929 de N.

## 7 · Comprobación de la entrega y cómo reproducirla

In [ ]:
# Las submissions: 1863 snapshots de test, cada uno una vez, 3 row_index válidos; coinciden con el top-3 de preds/
te_rows = pd.read_csv(comun.DATOS / "test.csv", usecols=["sample_id", "row_index"])
validas = set(map(tuple, te_rows.values))
for n in ("M30_30", "N30_ses", "C30_ses", "N10_ses", "C10_ses", "A", "B"):
    s = pd.read_csv(comun.SALIDAS / "submissions" / f"sub_{n}.csv")
    filas_ok = all((sid, int(r)) in validas for sid, p in zip(s.sample_id, s.prediction) for r in p.split())
    igual_preds = (comun.to_submission(leer(n, "preds")).set_index("sample_id").prediction
                   .reindex(s.sample_id).values == s.prediction.values).all()
    ok = (set(s.sample_id) == set(te_rows.sample_id) and s.sample_id.is_unique
          and s.prediction.str.split().str.len().eq(3).all() and filas_ok)
    print(f"sub_{n + '.csv':16s} {len(s)} filas · válida: {ok} · igual al top-3 de preds/{n}.csv: {igual_preds}")

**Reproducir la entrega** (desde la raíz del repo o desde `entrega/`; LightGBM con 14 hilos):

```
python entrega/codigo/entrenar_futuro.py N --rapido          # comprobación: fold 0, 1 semilla, no guarda (~1–2 min)
python entrega/codigo/entrenar_futuro.py N --semillas 30     # N30 y N30_ses (~1 h)
python entrega/codigo/entrenar_futuro.py C --semillas 30     # C30 y C30_ses (con el oráculo)
python entrega/codigo/entrenar_futuro.py mezcla              # M30_30 = 0.7·C30_ses + 0.3·N30_ses (imprime la CV anidada del peso)
```

- `codigo/features_futuro.py`: bloques `base`, `logic`, `past`, `future`, `clear`, `instr` (los mismos que
  `futuro/features_future.py`, `feat_clear.py` y `pruebas/f10_instr.py` del repo); caché en `entrega/cache/features_futuro.parquet`.
- `codigo/oraculo.py`: el oráculo de coordinaciones por fold; caché en `entrega/cache/oraculo_coord.parquet`.
- `codigo/sesion.py`: el segundo paso de sesión (logit condicional de 9 pesos).
- `codigo/entrenar_futuro.py`: N, C y la mezcla; guarda en `salidas/{oof,preds,submissions}`.
- `codigo/eleccion.py` y `codigo/sinpista.py`: las secciones 6b y 5b; no entrenan.

**Salidas que usa este notebook** (`entrega/salidas/`): `M30_30` (entrega), `N30_ses`, `C30_ses` (sus dos componentes), `N3/N10_ses`,
`C3/C10_ses`, `C10_otra_maquina_ses` (ruido de semillas y reproducibilidad), `futuro_base`, `futuro_clear`, `futuro_clear_ses`,
`A`, `B` (pasos de la cadena), `limpia` y `real_adri_noche` (líneas sin futuro).